In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yasa
import psutil
from psutil._common import bytes2human

from pathlib import Path
from scipy.io import loadmat
from mne.filter import resample
from itertools import chain
from tqdm import tqdm
from sys import getsizeof

from functions import get_sequences
from functions import phasic_rem_v3
from functions import create_name_cbd, create_name_rgs, create_name_os
from runtime_logger import logger_setup

CBD_DIR = "/home/nero/datasets/CBD/"
RGS_DIR = "/home/nero/datasets/RGS14/"
OS_DIR = "/home/nero/datasets/OSbasic/"

OUTPUT_DIR  = "/home/nero/datasets/preprocesssed/"
CBD_OVERVIEW_PATH = "overview.csv"

overview_df = pd.read_csv(CBD_OVERVIEW_PATH)

fs_cbd = 2500
fs_os = 2500
fs_rgs = 1000

targetFs = 500
n_down_cbd = fs_cbd/targetFs
n_down_rgs = fs_rgs/targetFs
n_down_os = fs_os/targetFs

phasic_tonic_idx = {}

phasic_tonic_freq = {
    'rat_id': [],
    'study_day': [],
    'condition': [],
    'treatment': [],
    'trial_num': [],
    'state': [],
    'count': [],
    'count_norm': [],
    'duration': []
    }

logger = logger_setup()

# CBD Dataset

In [2]:
pattern = r"[\w-]+posttrial[\w-]+"
mapped = {}

for root, dirs, fils in os.walk(CBD_DIR):
    for dir in dirs:
        # Check if the directory is a post trial directory
        if re.match(pattern, dir, flags=re.IGNORECASE):
            dir = Path(os.path.join(root, dir))
            HPC_file = next(dir.glob("*HPC*continuous*"))
            states = next(dir.glob('*-states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of trials: ", len(mapped))

Number of trials:  170


In [3]:
with tqdm(mapped.keys()) as t:
    for state in t:
        hpc = mapped[state]

        title = create_name_cbd(hpc, overview_df)
        logger.debug("Loading: {0}".format(title))
        logger.debug("fname: {0}".format(state))
        
        metadata = {}
        metaname_list  = title.split('_')
        metadata["rat_id"]    = int(metaname_list[0][3:])
        metadata["study_day"] = int(metaname_list[1][2:])
        metadata["condition"] = metaname_list[2]
        metadata["treatment"] = int(metaname_list[3])
        metadata["trial_num"] = int(metaname_list[4][-1])

        # Load the LFP data
        lfpHPC = loadmat(hpc)['HPC']
        lfpHPC = lfpHPC.flatten()

        # Load the states
        hypno = loadmat(state)['states']
        hypno = hypno.flatten()

        logger.debug("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))

        # Skip if no REM epoch is detected
        if(not (np.any(hypno == 5))):
            logger.debug("No REM detected. Skipping.")
            continue
        
        del hypno # Save memory

        #%% downsample the data
        logger.debug("Downsampling")
        data_resample = resample(lfpHPC, down=n_down_cbd, method='fft', verbose="INFO")
        logger.debug("Finished downsampling.")
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        
        del lfpHPC # Save memory

        # Remove artifacts
        logger.debug("Artifact removal.")
        art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
        art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
        data_resample[art_up] = 0

        del art_up # Save memory

        fname = OUTPUT_DIR + title
        np.savez(fname, data_resample)
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        logger.debug("Saving downsampled data ({0}) to {1}.".format(str(data_resample.shape), title))
        
        del data_resample # Save memory

        t.set_postfix_str(title)


 17%|█▋        | 29/170 [01:06<04:09,  1.77s/it, Rat5_SD3_OD_0_posttrial3] 

: 

In [4]:
state = "/home/nero/datasets/CBD/Rat5/Rat_OS_Ephys_cbd_chronic_Rat5_411358_SD3_OD_14-15_07_2021/2021-07-14_16-17-11_posttrial5/2021-07-14_16-17-11_posttrial5-states_ES.mat"

hpc = mapped[state]
title = create_name_cbd(hpc, overview_df)

# Load the LFP data
lfpHPC = loadmat(hpc)['HPC']
lfpHPC = lfpHPC.flatten()

# Load the states
hypno = loadmat(state)['states']
hypno = hypno.flatten()
    
print(bytes2human(psutil.virtual_memory().total), str(psutil.virtual_memory().percent))

data_resample = resample(lfpHPC, down=n_down_cbd, method='fft')

print(bytes2human(psutil.virtual_memory().total), str(psutil.virtual_memory().percent))

del lfpHPC

print(bytes2human(psutil.virtual_memory().total), str(psutil.virtual_memory().percent))

 # Remove artifacts
art_std, zscores_std = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
data_resample[art_up] = 0

fname = OUTPUT_DIR + title
np.savez(fname, data_resample)

7.7G 28.1
7.7G 54.1
7.7G 51.5


# RGS14 Dataset

In [4]:
pattern1 = r"[\w-]+post[\w-]+trial[\w-]+"
mapped = {}

for root, dirs, fils in os.walk(RGS_DIR):
    for dir in dirs:
        # Check if the directory is a post trial directory
        if re.match(pattern1, dir, flags=re.IGNORECASE):
            dir = Path(os.path.join(root, dir))
            HPC_file = next(dir.glob("*HPC*continuous*"))
            states = next(dir.glob('*-states*'))
            mapped[states] = HPC_file

print("Number of trials: ", len(mapped))

Number of trials:  166


In [5]:
for state in mapped.keys():
    hpc = mapped[state]

    title = create_name_rgs(str(hpc))
    print("Loading:", title)

    metadata = {}
    metaname_list  = title.split('_')
    metadata["rat_id"]    = int(metaname_list[0][3:])
    metadata["study_day"] = int(metaname_list[1][2:])
    metadata["condition"] = metaname_list[2]
    metadata["treatment"] = int(metaname_list[3])
    metadata["trial_num"] = int(metaname_list[4][-1])

    # Load the LFP data
    lfpHPC = loadmat(hpc)['HPC']
    lfpHPC = lfpHPC.flatten()
    
    # Load the states
    hypno = loadmat(state)['states']
    hypno = hypno.flatten()

    # Skip if no REM epoch is detected
    if(not (np.any(hypno == 5))):
        print("No REM detected. Skipping.")
        continue
    
    #%% downsample the data
    print("Downsampling")
    data_resample = resample(lfpHPC, down=n_down_rgs, method='fft')
    print("Finished downsampling.")

    del lfpHPC # Save memory

    seq = get_sequences(np.where(hypno==5)[0])

    print("Number of REM epochs:", len(seq))
    print("Resampled data shape:", data_resample.shape)
    print("Sleep states shape:", hypno.shape)
    
    phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
    ######### Save phasic/tonic REM indices #####################
    rem_idx = {}
    # Create a dict of start and end indices for each rem epoch
    for s in seq:
        rem_idx[s[0]] = s[-1]
    
    phasic = []
    tonic = []

    for rem_start in phasicREM:
        rem_end = rem_idx[rem_start]
        phasic += phasicREM[rem_start]
        
        tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
        tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
        print(rem_start*targetFs, rem_end*targetFs)

    idx = {}
    idx["phasic"] = phasic
    idx["tonic"] = tonic
    phasic_tonic_idx[title] = idx
    #############################################################
    
    print("-"*30)
    print("Phasic REM true indices:", phasic)
    print("Tonic REM true indices:", tonic)
    print("-"*30)

    print("*"*30)
    ###### Save phasic/tonic frequency and duration #############    
    dur_phasic = [(ep[1] - ep[0])/targetFs for ep in phasic]
    dur_tonic = [(ep[1] - ep[0])/targetFs for ep in tonic]
    
    cnt_phasic = len(dur_phasic)
    cnt_tonic = len(dur_tonic)  


    for state in ["phasic", "tonic"]:
        # Add metadata
        for condition in metadata.keys():
            phasic_tonic_freq[condition].append(metadata[condition])
        
        phasic_tonic_freq["state"].append(state)

        dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

        phasic_tonic_freq["duration"].append(np.sum(dur))
        phasic_tonic_freq["count"].append(len(dur))

        # Count normalized by duration
        if (metadata["trial_num"] == 5):
            phasic_tonic_freq["count_norm"].append(len(dur)/180)
        else:
            phasic_tonic_freq["count_norm"].append(len(dur)/45)

        if(state == "phasic"):
            print("Phasic REM duration:", dur)
        else:
            print("Tonic REM duration:", dur)
        
        print("Total:", phasic_tonic_freq["duration"][-1], "s")
        print("Count:", phasic_tonic_freq["count"][-1])
        print("Count normalized:", phasic_tonic_freq["count_norm"][-1])
        
        #per one REM epoch -> num_tonic = num_phasic + 1
        #total_num_tonic = num_phasic + num_rem_epochs
    #############################################################
    print("*"*30)
    break

Loading: Rat2_SD3_CON_2_posttrial1
Downsampling
Finished downsampling.
Number of REM epochs: 4
Resampled data shape: (1350230,)
Sleep states shape: (2699,)
967000 977500
1224500 1270000
------------------------------
Phasic REM true indices: [(973639, 974377), (1265187, 1265684)]
Tonic REM true indices: [(967000, 973639), (974377, 977500), (1224500, 1265187), (1265684, 1270000)]
------------------------------
******************************
Phasic REM duration: [1.476, 0.994]
Total: 2.4699999999999998 s
Count: 2
Count normalized: 0.044444444444444446
Tonic REM duration: [13.278, 6.246, 81.374, 8.632]
Total: 109.53 s
Count: 4
Count normalized: 0.08888888888888889
******************************


# OSbasic Dataset

In [6]:
pattern2 = r".*post_trial.*"
mapped = {}

for root, dirs, fils in os.walk(OS_DIR):
    for dir in dirs:
        dir = (os.path.join(root, dir))
        # Check if the directory is a post trial directory
        if re.match(pattern2, dir, flags=re.IGNORECASE):
            dir = Path(dir)
            HPC_file = next(dir.glob("*HPC*"))
            states = next(dir.glob('*states*'))
            mapped[states] = HPC_file

print("Number of trials: ", len(mapped))


Number of trials:  210


In [7]:
for state in mapped.keys():
    hpc = mapped[state]

    title = create_name_os(str(hpc))
    print("Loading:", title)

    metadata = {}
    metaname_list  = title.split('_')
    metadata["rat_id"]    = int(metaname_list[0][3:])
    metadata["study_day"] = int(metaname_list[1][2:])
    metadata["condition"] = metaname_list[2]
    metadata["treatment"] = int(metaname_list[3])
    metadata["trial_num"] = int(metaname_list[4][-1])

    # Load the LFP data
    lfpHPC = loadmat(hpc)['HPC']
    lfpHPC = lfpHPC.flatten()
    
    # Load the states
    hypno = loadmat(state)['states']
    hypno = hypno.flatten()

    # Skip if no REM epoch is detected
    if(not (np.any(hypno == 5))):
        print("No REM detected. Skipping.")
        continue
    
    #%% downsample the data
    print("Downsampling")
    data_resample = resample(lfpHPC, down=n_down_os, method='fft')
    print("Finished downsampling.")

    del lfpHPC # Save memory

    seq = get_sequences(np.where(hypno==5)[0])

    print("Number of REM epochs:", len(seq))
    print("Resampled data shape:", data_resample.shape)
    print("Sleep states shape:", hypno.shape)
    
    phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
    ######### Save phasic/tonic REM indices #####################
    rem_idx = {}
    # Create a dict of start and end indices for each rem epoch
    for s in seq:
        rem_idx[s[0]] = s[-1]
    
    phasic = []
    tonic = []

    for rem_start in phasicREM:
        rem_end = rem_idx[rem_start]
        phasic += phasicREM[rem_start]
        
        tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
        tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
        print(rem_start*targetFs, rem_end*targetFs)

    idx = {}
    idx["phasic"] = phasic
    idx["tonic"] = tonic
    phasic_tonic_idx[title] = idx
    #############################################################
    
    print("-"*30)
    print("Phasic REM true indices:", phasic)
    print("Tonic REM true indices:", tonic)
    print("-"*30)

    print("*"*30)
    ###### Save phasic/tonic frequency and duration #############    
    dur_phasic = [(ep[1] - ep[0])/targetFs for ep in phasic]
    dur_tonic = [(ep[1] - ep[0])/targetFs for ep in tonic]
    
    cnt_phasic = len(dur_phasic)
    cnt_tonic = len(dur_tonic)  

    
    for state in ["phasic", "tonic"]:
        # Add metadata
        for condition in metadata.keys():
            phasic_tonic_freq[condition].append(metadata[condition])
        
        phasic_tonic_freq["state"].append(state)

        dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

        phasic_tonic_freq["duration"].append(np.sum(dur))
        phasic_tonic_freq["count"].append(len(dur))

        # Count normalized by duration
        if (metadata["trial_num"] == 5):
            phasic_tonic_freq["count_norm"].append(len(dur)/180)
        else:
            phasic_tonic_freq["count_norm"].append(len(dur)/45)

        if(state == "phasic"):
            print("Phasic REM duration:", dur)
        else:
            print("Tonic REM duration:", dur)
        
        print("Total:", phasic_tonic_freq["duration"][-1], "s")
        print("Count:", phasic_tonic_freq["count"][-1])
        print("Count normalized:", phasic_tonic_freq["count_norm"][-1])

        #per one REM epoch -> num_tonic = num_phasic + 1
        #total_num_tonic = num_phasic + num_rem_epochs
    #############################################################
    print("*"*30)
    break

Loading: Rat1_SD1_OD_4_posttrial5
Downsampling
Finished downsampling.
Number of REM epochs: 15
Resampled data shape: (5414067,)
Sleep states shape: (10827,)
597000 717500
1276000 1315500
1338000 1378000
2084000 2227000
2545000 2562500
2943000 3026500
4001000 4017000
4314000 4366000
5102000 5242500
------------------------------
Phasic REM true indices: [(614210, 614766), (640372, 640866), (643938, 644549), (672881, 673536), (685909, 686891), (694810, 695836), (1291913, 1292671), (1305160, 1305714), (1361531, 1361982), (2099554, 2100202), (2143608, 2145752), (2155834, 2157130), (2182046, 2183636), (2194104, 2194596), (2212620, 2213287), (2219957, 2220789), (2558892, 2561720), (2974731, 2975459), (2993192, 2996200), (3018293, 3019032), (4005603, 4006159), (4329308, 4329781), (4335620, 4336290), (4336598, 4337649), (4347495, 4348013), (4356781, 4357345), (5129892, 5131422), (5164066, 5164919)]
Tonic REM true indices: [(597000, 614210), (614766, 640372), (640866, 643938), (644549, 672881),

# Save

In [8]:
np.save('phasic_tonic_idx', phasic_tonic_idx)
# For loading use
# data = np.load('data.npy', allow_pickle='TRUE').item()

ep_df = pd.DataFrame({key:pd.Series(value) for key, value in phasic_tonic_freq.items()})
ep_df.to_csv("states_freq_dur.csv", index=False)

In [9]:
ep_df

,rat_id,study_day,condition,treatment,trial_num,state,count,count_norm,duration
0,5,8,HC,0,3,phasic,4,0.088889,10.270
1,5,8,HC,0,3,tonic,6,0.133333,190.730
2,2,3,CON,2,1,phasic,2,0.044444,2.470
3,2,3,CON,2,1,tonic,4,0.088889,109.530
4,1,1,OD,4,5,phasic,28,0.155556,54.548
5,1,1,OD,4,5,tonic,37,0.205556,1250.452
